# Day 2 Session 3 -- Exercises: LangGraph Orchestration & Workflow Design

Work through the task below to practice building LangGraph workflows. You have the demos notebook as reference.

**Engineering context:** You are an engineer building workflow orchestration systems. Your users (consultants) need reliable, stateful pipelines that route work intelligently, refine outputs iteratively, and include human checkpoints. LangGraph lets you model these requirements as directed graphs with typed state, conditional edges, and cycles.

In [1]:
!pip install -q langchain langchain-openai langchain-core langgraph python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


## Setup

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, Literal
import operator
import json
import os

print("All imports successful!")

All imports successful!


## Recap

In the demos you saw five LangGraph patterns: linear pipelines with typed state and sequential edges, conditional routing that branches based on LLM classification, the ReAct think-act-observe loop, cyclic refinement that iterates until quality thresholds are met, and human-in-the-loop gates for approval checkpoints. This exercise focuses on the foundational pattern -- building a linear StateGraph pipeline.

---
## Task 1: Client Onboarding Pipeline (3-Node Linear StateGraph)

**Scenario:** When a new client engagement kicks off, the team runs a standard onboarding pipeline:
1. **gather_data** -- Clean and normalize the raw client data (remove extra whitespace)
2. **analyze_situation** -- Use an LLM to generate a preliminary diagnostic summary highlighting key challenges and opportunities (2-3 sentences)
3. **prepare_brief** -- Use an LLM to transform the diagnostic into an executive-ready brief suitable for partner review (2-3 sentences)

**Your task:** Build a three-node linear StateGraph that processes raw client data through these three steps.

**Hints:**
- Define an `OnboardingState` TypedDict with fields: `raw_client_data`, `clean_data`, `diagnostic_summary`, `executive_brief`
- `gather_data` is pure Python (no LLM needed) -- just clean the whitespace with `" ".join(state["raw_client_data"].split())`
- `analyze_situation` and `prepare_brief` each call `llm.invoke()` with a SystemMessage and HumanMessage
- Use `graph.set_entry_point()`, `graph.add_edge()`, and connect the last node to `END`

In [3]:
# Task 1 - Client Onboarding Pipeline

# TODO: Define OnboardingState TypedDict with four fields:
#   raw_client_data, clean_data, diagnostic_summary, executive_brief


# TODO: Initialize the LLM


# TODO: Define gather_data node -- clean/normalize raw_client_data


# TODO: Define analyze_situation node -- LLM diagnostic summary


# TODO: Define prepare_brief node -- LLM executive brief


# TODO: Build the StateGraph, add nodes, set entry point, add edges, compile


# TODO: Invoke with the test data below and print results
# result = app.invoke({
#     "raw_client_data": "  Client: GlobalRetail Corp.   Revenue $8.2B,   declining   margins   from 12% to 7%   over 3 years.    Major  cost  pressures  in  supply chain   and  labor.  Considering  digital  transformation   but  lacks  internal  capabilities.   Competitors  gaining  share  in   e-commerce  channel.  ",
#     "clean_data": "", "diagnostic_summary": "", "executive_brief": ""
# })
# print(f"Clean Data: {result['clean_data']}")
# print(f"\nDiagnostic: {result['diagnostic_summary']}")
# print(f"\nExecutive Brief: {result['executive_brief']}")

### Expected Output

When you run the cell above with the provided test data, you should see output similar to:

```
Clean Data: Client: GlobalRetail Corp. Revenue $8.2B, declining margins from 12% to 7% over 3 years. Major cost pressures in supply chain and labor. Considering digital transformation but lacks internal capabilities. Competitors gaining share in e-commerce channel.

Diagnostic: GlobalRetail Corp. faces a margin compression challenge driven by rising supply chain
and labor costs, compounded by competitive losses in the e-commerce channel. The company's interest
in digital transformation is well-placed but will require external capability building given the
current internal gap. Priority areas for investigation include supply chain cost reduction
opportunities, digital channel strategy, and build-vs-buy decisions for technology capabilities.

Executive Brief: GlobalRetail Corp., an $8.2B retailer experiencing significant margin erosion (12%
to 7% over three years), requires an integrated diagnostic spanning supply chain optimization,
digital transformation readiness, and competitive response strategy. We recommend a 10-week
engagement to quantify cost reduction levers and develop a phased digital transformation roadmap
with clear capability-building milestones.
```

The exact wording will vary (LLM output is non-deterministic), but you should see:
- **Clean Data** with normalized whitespace (no extra spaces)
- **Diagnostic** summarizing challenges and opportunities in 2-3 sentences
- **Executive Brief** in polished consulting language suitable for partner review